#### Environment test {install & import}

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

load_dotenv()
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

Groq API key loaded: True


##### Load Data

In [7]:
# Cell 2 (UPDATED): Load FULL CSV
import pandas as pd

# Load ENTIRE dataset (not just 5 rows)
df = pd.read_csv("../data/true.csv")

print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
display(df.head(3))

# Check memory usage
print(f"\nDataFrame shape: {df.shape}")
print(f"Estimated size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 21417
Columns: ['title', 'text', 'subject', 'date']

First 3 rows:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"



DataFrame shape: (21417, 4)
Estimated size: 82.65 MB


##### Text Cleaning & Chunking

In [8]:
# Cell 3 (FINAL): Production-Ready Chunking
import re
from typing import List, Tuple

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^\w\s.]", " ", text)
    return " ".join(text.split())

def chunk_documents_optimized(
    df: pd.DataFrame,
    text_col: str = "text",
    metadata_cols: List[str] = None,
    chunk_size: int = 300,      # OPTIMAL: 256-512 tokens range
    overlap: int = 50,          # 15-20% overlap
    min_chunk_words: int = 20   # Reasonable minimum
) -> Tuple[List[str], List[dict]]:
    if metadata_cols is None:
        metadata_cols = ["title", "subject", "date"]
    
    chunks = []
    metadata = []
    
    total_rows = len(df)
    print(f"Processing {total_rows} rows...")
    
    for idx, row in df.iterrows():
        if idx % 5000 == 0:
            print(f"  Progress: {idx}/{total_rows} rows processed")
        
        full_text = clean_text(str(row[text_col]))
        words = full_text.split()
        
        for start in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[start:start+chunk_size])
            if len(chunk.split()) < min_chunk_words:
                continue
            
            chunks.append(chunk)
            meta = {"source_row": int(idx), "chunk_id": len(chunks)}
            for col in metadata_cols:
                if col in df.columns:
                    meta[col] = str(row[col])
            metadata.append(meta)
    
    print(f"Chunking complete!")
    return chunks, metadata

# Run optimized chunking
text_column = "text"
metadata_columns = ["title", "subject", "date"]

print("="*60)
print("Starting chunking process...")
print("="*60)

chunks, metadata = chunk_documents_optimized(df, text_col=text_column, metadata_cols=metadata_columns, chunk_size=300, overlap=50, min_chunk_words=20)

print(f"\n{'='*60}")
print(f"Total chunks created: {len(chunks)}")
print(f"Total metadata entries: {len(metadata)}")
print(f"Average chunks per row: {len(chunks)/len(df):.2f}")
print(f"{'='*60}")

Starting chunking process...
Processing 21417 rows...
  Progress: 0/21417 rows processed
  Progress: 5000/21417 rows processed
  Progress: 10000/21417 rows processed
  Progress: 15000/21417 rows processed
  Progress: 20000/21417 rows processed
Chunking complete!

Total chunks created: 43516
Total metadata entries: 43516
Average chunks per row: 2.03


##### Better Embedding Model

In [10]:
from sentence_transformers import SentenceTransformer

# Use faster model (good accuracy, 3x faster)
model_name = "all-MiniLM-L6-v2"  # Faster than mpnet
print(f"Loading model: {model_name}...")
model = SentenceTransformer(model_name)

print(f"\nCreating embeddings for {len(chunks)} chunks...")
print("(This will take 5-10 minutes for 43k chunks)")
embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True, batch_size=128)
print(f"Embeddings shape: {embeddings.shape}")

Loading model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Creating embeddings for 43516 chunks...
(This will take 5-10 minutes for 43k chunks)


Batches:   0%|          | 0/340 [00:00<?, ?it/s]

Embeddings shape: (43516, 384)


##### FAISS Index & Save

In [13]:
import faiss
import pickle
import json
import os

print("Creating FAISS index...")
faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Saving to disk...")
faiss.write_index(index, "faiss_index.bin")
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Get file sizes
index_size = os.path.getsize("faiss_index.bin") / 1024**2
chunks_size = os.path.getsize("chunks.pkl") / 1024**2
metadata_size = os.path.getsize("metadata.json") / 1024**2

print(f"\n{'='*60}")
print(f"FAISS index saved successfully!")
print(f"Indexed {len(chunks)} chunks, dim={dim}")
print(f"Index size: {index_size:.2f} MB")
print(f"Chunks size: {chunks_size:.2f} MB")
print(f"Metadata size: {metadata_size:.2f} MB")
print(f"Total: {index_size + chunks_size + metadata_size:.2f} MB")
print(f"{'='*60}")

Creating FAISS index...
Saving to disk...

FAISS index saved successfully!
Indexed 43516 chunks, dim=384
Index size: 63.74 MB
Chunks size: 54.20 MB
Metadata size: 8.55 MB
Total: 126.49 MB


##### Test Retrieval Function

In [14]:
def retrieve(query: str, top_k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "chunk": chunks[idx],
            "score": float(score),
            "metadata": metadata[idx]
        })
    return results

# Test
test_query = "What is Republican budget policy?"
results = retrieve(test_query, top_k=5)

print(f"Query: {test_query}")
print(f"{'='*60}")
for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {r['score']:.3f}) ---")
    print(f"Title: {r['metadata'].get('title', 'N/A')}")
    print(f"Subject: {r['metadata'].get('subject', 'N/A')}")
    print(f"Chunk: {r['chunk'][:250]}...")

Query: What is Republican budget policy?

--- Result 1 (score: 0.605) ---
Title: Paul Ryan sees common ground with Trump budget plan
Subject: politicsNews
Chunk: WASHINGTON Reuters The U.S. House of Representatives Republican leaders on Tuesday praised President Donald Trump s proposed federal spending budget and said lawmakers would be able to find common ground with the administration s plan. At least we no...

--- Result 2 (score: 0.599) ---
Title: Obama proposes $4.1 trillion spending plan in final White House budget
Subject: politicsNews
Chunk: WASHINGTON Reuters U.S. President Barack Obama proposed a 4.1 trillion spending plan for fiscal year 2017 on Tuesday in a final White House budget that met immediate Republican resistance for its cost and reliance on tax hikes to fund domestic priori...

--- Result 3 (score: 0.588) ---
Title: Trump budget on the menu as U.S. senators lunch with Tillerson
Subject: politicsNews
Chunk: block deliveries of food and medical care. Many of Trump s

##### Reranking

In [15]:
# Cell 7: Reranking with Cross-Encoder
from sentence_transformers import CrossEncoder

print("Loading cross-encoder for reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_with_rerank(query: str, top_k: int = 10, rerank_top_k: int = 5):
    # First retrieve top-k
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    
    # Prepare candidates
    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        candidates.append({
            "chunk": chunks[idx],
            "initial_score": float(score),
            "metadata": metadata[idx]
        })
    
    # Rerank
    if len(candidates) > 0:
        pairs = [[query, c["chunk"]] for c in candidates]
        rerank_scores = cross_encoder.predict(pairs)
        
        # Sort by rerank score
        for i, score in enumerate(rerank_scores):
            candidates[i]["rerank_score"] = float(score)
        
        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
        return candidates[:rerank_top_k]
    
    return []

# Test reranking
test_query = "What is Republican budget policy?"
reranked_results = retrieve_with_rerank(test_query, top_k=10, rerank_top_k=5)

print(f"\nReranked Results for: {test_query}")
print(f"{'='*60}")
for i, r in enumerate(reranked_results, 1):
    print(f"\n--- Reranked Result {i} (score: {r['rerank_score']:.3f}) ---")
    print(f"Title: {r['metadata'].get('title', 'N/A')}")
    print(f"Chunk: {r['chunk'][:250]}...")

Loading cross-encoder for reranking...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Reranked Results for: What is Republican budget policy?

--- Reranked Result 1 (score: 2.053) ---
Title: Republicans see tax reform complicated by Trump deal with Democrats
Chunk: year 2018 budget resolution that is critical to the Republican strategy for tax reform. The budget contains a procedural rule that would allow Republicans to enact tax legislation with a simple majority in the Senate which they control by a 52 48 mar...

--- Reranked Result 2 (score: 1.241) ---
Title: Paul Ryan sees common ground with Trump budget plan
Chunk: WASHINGTON Reuters The U.S. House of Representatives Republican leaders on Tuesday praised President Donald Trump s proposed federal spending budget and said lawmakers would be able to find common ground with the administration s plan. At least we no...

--- Reranked Result 3 (score: 0.612) ---
Title: House Republicans float controversial budget plan
Chunk: WASHINGTON Reuters U.S. House Republicans on Tuesday insisted they will proceed with a fiscal 201

#### Comprehensive Evaluation {Multiple Queries}

In [17]:
# Cell 8 Evaluation with Multiple Test Queries
from evaluate import load
import time
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

# Initialize LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

def generate_answer(question: str, contexts: list[str]) -> str:
    context_text = "\n\n---\n\n".join(contexts)
    prompt = ChatPromptTemplate.from_template(
        "You are a helpful Q&A assistant. Answer ONLY using the context below. "
        "If the context does not contain the answer, say 'I don't know based on the provided context.'\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n"
        "Answer:"
    )
    chain = prompt | llm
    response = chain.invoke({"question": question, "context": context_text})
    return response.content if hasattr(response, "content") else str(response)

# Load metrics
rouge = load("rouge")
bertscore = load("bertscore")

# Define test queries
test_queries = [
    "What is Republican budget policy?",
    "What does Trump say about Amazon?",
    "What are education spending priorities?",
    "What is the news about tax cuts?",
    "What did Republicans say about fiscal conservative?",
    "What is the latest political news?",
    "What are infrastructure spending plans?",
    "What is Congress discussing?",
    "What are the healthcare policy changes?",
    "What is the economic outlook?",
]

# Store results
all_results = []
all_retrieval_scores = []

print(f"Running evaluation on {len(test_queries)} queries...")
print("="*60)

for i, q in enumerate(test_queries, 1):
    print(f"\nQuery {i}/{len(test_queries)}: {q}")
    
    # Measure retrieval time
    start_time = time.time()
    retrieved = retrieve_with_rerank(q, top_k=10, rerank_top_k=5)
    retrieval_time = time.time() - start_time
    
    contexts = [r["chunk"] for r in retrieved]
    answer = generate_answer(q, contexts)
    
    top_score = retrieved[0]["rerank_score"] if retrieved else 0
    all_results.append({
        "query": q,
        "answer": answer,
        "top_score": top_score,
        "retrieval_time": retrieval_time,
        "sources": retrieved
    })
    all_retrieval_scores.append(top_score)
    
    print(f"  Top score: {top_score:.3f}")
    print(f"  Retrieval time: {retrieval_time:.3f}s")
    print(f"  Answer: {answer[:100]}...")

# Summary statistics
print(f"\n{'='*60}")
print("EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"Total queries: {len(test_queries)}")
print(f"Average retrieval score: {sum(all_retrieval_scores)/len(all_retrieval_scores):.3f}")
print(f"Min score: {min(all_retrieval_scores):.3f}")
print(f"Max score: {max(all_retrieval_scores):.3f}")
print(f"Average retrieval time: {sum(r['retrieval_time'] for r in all_results)/len(all_results):.3f}s")
print(f"Total chunks indexed: {len(chunks)}")
print(f"Embedding model: all-MiniLM-L6-v2")
print(f"Reranking model: cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"{'='*60}")

Running evaluation on 10 queries...

Query 1/10: What is Republican budget policy?
  Top score: 2.053
  Retrieval time: 1.395s
  Answer: I don't know based on the provided context....

Query 2/10: What does Trump say about Amazon?
  Top score: 8.314
  Retrieval time: 1.020s
  Answer: Trump said that Amazon has a huge antitrust problem and that it is getting away with murder tax-wise...

Query 3/10: What are education spending priorities?
  Top score: -3.759
  Retrieval time: 0.937s
  Answer: Based on the provided context, education spending priorities mentioned include:

1. Increasing fundi...

Query 4/10: What is the news about tax cuts?
  Top score: 2.791
  Retrieval time: 0.885s
  Answer: The news about tax cuts is that the White House is planning to release the biggest tax cut in U.S. h...

Query 5/10: What did Republicans say about fiscal conservative?
  Top score: 6.284
  Retrieval time: 0.915s
  Answer: It's interesting to hear Mark talk about fiscal responsibility....

Query 6/